# StreamClipper ? Smart Hook & Payoff Vertical Shorts (Max 50s)
Turn long streams into **25s?50s** high-clarity (**1080p+**) vertical `1080x1920` Shorts with a real **hook + payoff**, styled captions, `clips.csv`, and `report/index.html`.
1. **Runtime > Change runtime type > T4 GPU**.
2. Run cells top to bottom; everything saves to `MyDrive/StreamClipper/` and resumes if interrupted.

In [ ]:
#@title 1. Check GPU & Mount Google Drive
import shutil, subprocess, os
from google.colab import drive

if shutil.which('nvidia-smi'):
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
    print('GPU:', gpu.stdout.strip())
    HAS_GPU = True
else:
    print('No GPU detected; running on CPU.')
    HAS_GPU = False

drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/StreamClipper', exist_ok=True)

In [ ]:
#@title 2. Install Engine & Dependencies
import os, subprocess

if subprocess.run(['which', 'ffmpeg'], capture_output=True).returncode != 0:
    subprocess.run(['apt-get', '-qq', 'update'], check=False)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'ffmpeg'], check=False)

!curl -fsSL https://deno.land/install.sh | sh > /dev/null 2>&1 || true
os.environ['PATH'] += ':/root/.deno/bin:/usr/local/bin'
!rm -rf hype-clipper && git clone https://github.com/Aromalajith8/hype-clipper.git hype-clipper
!pip install -q -r hype-clipper/requirements.txt
print('Engine ready!')

In [ ]:
#@title 3. Stream Source & Free AI Setup
source = '' #@param {type:"string"}
cookies_path = '/content/drive/MyDrive/cookies.txt' #@param {type:"string"}
ai_provider = 'auto' #@param ['auto', 'groq', 'gemini', 'ollama', 'heuristic']
ollama_model = 'qwen3.5:9b' #@param ['qwen3.5:9b', 'gemma4:12b']
groq_api_key = '' #@param {type:"string"}
gemini_api_key = '' #@param {type:"string"}

In [ ]:
#@title 4. Clip Length (25s-50s), 1080p Clarity & Caption/Hook Styles
min_height = 1080 #@param [1080, 1440, 2160]
min_len = 25 #@param {type:"slider", min:20, max:40, step:5}
max_len = 50 #@param {type:"slider", min:25, max:50, step:5}
max_clips = 25 #@param {type:"slider", min:5, max:60, step:5}
caption_style = 'auto' #@param ['auto', 'bold-pop', 'beast-bold', 'karaoke-highlight', 'word-by-word', 'two-line-stack', 'highlight-box', 'neon-glow', 'impact-shake', 'emoji-pop', 'typewriter', 'bounce', 'gradient-stroke', 'comic-burst', 'minimal-clean', 'gaming-glitch', 'none']
hook_style = 'bold-box' #@param ['bold-box', 'outline-glow', 'top-bar', 'none']
split_cam = False #@param {type:"boolean"}

In [ ]:
#@title 5. Run Smart Hook+Payoff Generator
import glob, os, shutil, subprocess, time

assert source.strip(), 'Paste a stream URL or Drive video path in Step 3 first!'

if groq_api_key.strip():
    os.environ['GROQ_API_KEY'] = groq_api_key.strip()
if gemini_api_key.strip():
    os.environ['GEMINI_API_KEY'] = gemini_api_key.strip()

if ai_provider == 'ollama' or (ai_provider == 'auto' and not groq_api_key.strip() and not gemini_api_key.strip()):
    if not shutil.which('ollama'):
        subprocess.run(['apt-get', '-qq', 'update'], check=False)
        subprocess.run(['apt-get', '-qq', 'install', '-y', 'zstd', 'curl'], check=False)
        subprocess.run('curl -fsSL -L https://github.com/ollama/ollama/releases/latest/download/ollama-linux-amd64.tgz -o /tmp/ollama.tgz && mkdir -p /usr/local && tar -xzf /tmp/ollama.tgz -C /usr/local && rm -f /tmp/ollama.tgz', shell=True, check=False)
    ollama_bin = shutil.which('ollama') or '/usr/local/bin/ollama'
    if os.path.exists(ollama_bin):
        subprocess.Popen([ollama_bin, 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(3)
        subprocess.run([ollama_bin, 'pull', ollama_model], check=False)

cmd = [
    'python3', 'hype-clipper/clipper.py', source.strip(),
    '--clips', str(max_clips),
    '--min-len', str(min_len),
    '--max-len', str(min(max_len, 50)),
    '--min-height', str(max(1080, min_height)),
    '--caption-style', caption_style,
    '--hook-style', hook_style,
    '--out', '/content/drive/MyDrive/StreamClipper',
    '--ai-provider', ai_provider,
    '--ollama-model', ollama_model,
]

if split_cam:
    cmd.append('--split-cam')
if cookies_path.strip() and os.path.exists(cookies_path.strip()):
    cmd += ['--cookies', cookies_path.strip()]

subprocess.run(cmd)
latest_sessions = sorted(glob.glob('/content/drive/MyDrive/StreamClipper/*'), key=os.path.getmtime)
LAST_SESSION_DIR = latest_sessions[-1] if latest_sessions else None
print('Latest session folder:', LAST_SESSION_DIR)

In [ ]:
#@title 6. Preview Top Clips & Report Summary
import glob, os
from IPython.display import Video, display

sess = globals().get('LAST_SESSION_DIR')
if not sess or not os.path.isdir(sess):
    sessions = sorted(glob.glob('/content/drive/MyDrive/StreamClipper/*'), key=os.path.getmtime)
    sess = sessions[-1] if sessions else None

if sess:
    print('Session:', sess)
    print('Report:', os.path.join(sess, 'report', 'index.html'))
    mp4s = sorted(glob.glob(os.path.join(sess, 'shorts', '*.mp4')))[:3]
    for m in mp4s:
        print('Preview:', os.path.basename(m))
        display(Video(m, embed=True, width=320))
else:
    print('No completed session found yet. Run Step 5 first.')